## Import of Libraries

In [1]:
# Import visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Import data handling libraries
import pandas as pd
import numpy as np
from numpy.linalg import norm
import pickle

# Import machine learning libraries
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

KeyboardInterrupt: 

## Configuration of Display Options

In [ ]:
# Set display format for floats to 2 decimal places
pd.set_option("display.float_format", "{:,.2f}".format)

## Loading of Dataset

In [ ]:
# Import filtered DataFrame
with open("../data/processed/df_filtered.pkl", "rb") as f:
    df_filtered = pickle.load(f)

# Import preprocessed DataFrame
with open("../data/processed/df_preprocessed.pkl", "rb") as f:
    df_preprocessed = pickle.load(f)

In [ ]:
# Plot names of preporcessed features
display("Preprocessed Features")
feature_names = df_preprocessed.columns.tolist()
feature_names

'Preprocessed Features'

NameError: name 'df_preprocessed' is not defined

## Baseline Model: k-Means

### Instantiate Model

In [ ]:
# Instantiate basline model (with high numer of classifications)
model_km = KMeans(
    n_clusters=3000,
    random_state=42
)

### Organize Data in Feature Matrix

In [ ]:
# Select columns for feature selection
selected_exact = [
    "raw__danceability",
    "raw__energy",
    "remainder__speechiness",
    "raw__instrumentalness",
    "raw__valence",
    "standard__tempo",
]

camelot_prefix = "cat__camelot__id_"

# Create masks for filtering data
mask_exact = np.isin(feature_names, selected_exact)
mask_camelot = np.char.startswith(feature_names.astype(str), camelot_prefix)

mask = mask_exact | mask_camelot

# Perform feature selection
df_features_selected = df_preprocessed.loc[:, mask]

df_features_selected

NameError: name 'np' is not defined

### Adjust Model to Data

In [ ]:
# Fit model
model_km.fit(df_features_selected)

In [ ]:
# Evaluate quality of selected number of clusters using WCSS (Within-cluster sum of squares)
display("WCSS (Within-cluster sum of squares)")
model_km.score(df_features_selected)

In [ ]:
# Predict classifications and add labels
labels_km = model_km.labels_  # prediction
df_filtered["label_km"] = labels_km

# Show value count 
display("Value Counts of Labels")
df_filtered["label_km"].value_counts()

In [ ]:
# Determine the Euclidean distances
distances_train = model_km.transform(df_features_selected) # Alternative: distances = euclidean_distances(X_train, model_km.cluster_centers_)

### Create Track Recommendations

In [ ]:
# Create recommendations based on a selected track
row    = 73498     
n_max  = 10

pos_ref = df_features_selected.index.get_loc(row)
cluster = labels_km[pos_ref]

mask_cluster   = (labels_km == cluster)
cluster_labels = df_features_selected.index[mask_cluster].tolist()

others = [i for i in cluster_labels if i != row] # Skip self-match
cluster_labels = [row] + others

subset = [row] + others[:n_max]

print(f"Cluster-ID (k-means): {cluster}")
print(f"Number of similar Tracks: {len(others)}")

display(df_filtered.loc[subset])

### Evaluation of Model Quality

In [ ]:
# Determining k: Elbow-Method
# Find the best number of clusters
cluster_scores = []
for n_cluster in range(1, 50, 2):
    model = KMeans(n_clusters=n_cluster, random_state=42)
    model.fit(df_features_selected)
    cluster_scores.append(model.score(df_features_selected))

display("Finding the best value for k")
fig, ax = plt.subplots()
ax.plot(range(1, 50, 2), 
        cluster_scores, 
        marker='o', 
        markersize=5,
        color="steelblue")
ax.set_xlabel("k")
ax.set_ylabel("Within-cluster sum of squares")

plt.tight_layout()

In [ ]:
# Interpretation of clusters
display("Mean values and standard deviations for different n_init")

for n_init in [1, 10, 20, 30, 40]:
    cluster_scores = []
    for i in range(10):
        model = KMeans(
            n_clusters=7, 
            n_init=n_init)
        model.fit(df_features_selected)
        cluster_scores.append(model.score(df_features_selected))
    print(np.mean(cluster_scores), np.std(cluster_scores))

In [ ]:
# Determine Silhouette Score
display("Silhouette Score")
silhouette_score(
    X=df_features_selected, 
    labels=model_km.labels_,
    metric='euclidean'
)

In [ ]:
# Determine Silhouette coefficients for k-Means
arr_sil = silhouette_samples(
    X=df_features_selected, 
    labels=model.labels_, 
    metric="euclidean"
)

display("Silhouette coefficients for k-Means with k=7")

fig, ax = plt.subplots(figsize=(15, 15))
start = 0
end = 0

for cluster in np.unique(model.labels_):
    mask = model.labels_ == cluster
    sv_len = len(arr_sil[mask])
    sv_sorted = np.sort(arr_sil[mask])
#    if any(x < 0 for x in sv_sorted) == False: 
#        print('Cluster:', cluster)
    end = end + sv_len
    ax.barh(range(start, end), width=sv_sorted, label = cluster)
    start = end

ax.legend()
plt.show()

### Insights:

Although kMeans only suggests seven clusters, it also generates good recommendations even with a very large number of clusters.

## Additional Model: DBSCAN

### Instantiate Model

In [ ]:
# Instantiate additional 
model_db = DBSCAN(
    eps=0.1,
    min_samples=12  
)

### Organize Data in Feature Matrix

In [ ]:
df_features_selected

### Adjust Model to Data

In [ ]:
# Fit model
model_db.fit(df_features_selected)

In [ ]:
# Predict classifications and add labels
labels_db = model_db.labels_  # prediction
df_filtered["label_db"] = labels_db

# Show value count 
display("Value Counts of Labels")
df_filtered["label_db"].value_counts()

### Create Track Recommendations

In [ ]:
# Create recommendations based on a selected track
row  = 73498
n_max  = 10

pos_ref   = df_features_selected.index.get_loc(row)
cluster   = labels_db[pos_ref]

if cluster == -1:
    print("Reference track is in noise (-1). There is no corresponding Cluster.")
else:
    mask_cluster   = (labels_db == cluster)
    cluster_labels = df_features_selected.index[mask_cluster].tolist()
    
    cluster_labels = [row] + [i for i in cluster_labels if i != row] # Skip self-match
    subset = cluster_labels[: (1 + n_max)]

    print(f"Cluster-ID (DBSCAN): {cluster}")
    print(f"Number of similar Tracks: {len(cluster_labels)}")

    display(df_filtered.loc[subset])


### Evaluation of Model Quality

In [ ]:
# Determine distances between data points
arr_dist = euclidean_distances(df_features_selected)
arr_dist_sorted = np.sort(arr_dist, axis=1)

# Plot distribution of distances to the nearest neighbor
display("Distribution of Distances to the Nearest Neighbor")
sns.displot(arr_dist_sorted[:, 1], kde=False);

In [ ]:
# Create a k-distance plot (Distances to nearest neighbors)
display("Sorted Distances to 12th Neighbor")
fig, ax = plt.subplots()

ax.plot(range(len(arr_dist_sorted)), np.sort(arr_dist_sorted[:, 12]))  # min_sampels = 2 x num_of_dimensions
ax.set(xlabel='Data points', ylabel='Distance to 12th Neighbor');

In [ ]:
# Determine maximum distance eps
np.sort(arr_dist_sorted[:, 12])[17000]

In [ ]:
# Instantiate model with optimized hyperparameters
model = DBSCAN(
    eps=0.29, 
    min_samples=12)

# Fit model
model.fit(df_features_selected)

# Clusternamen
np.unique(model.labels_)

In [ ]:
# Calculate sum of outliers
np.sum(model.labels_ == -1)

In [ ]:
# Show cluster assignment (dimensionality reduced)
#pca
pca = PCA(n_components=2)
cluster_plot_arr = pca.fit_transform(df_features_selected)

#plot
display("Cluster Assignment (Dimensionality reduced)")
ax = sns.scatterplot(x=cluster_plot_arr[:,0],
                     y=cluster_plot_arr[:,1],
                     alpha=0.7,
                     hue=model.labels_)

#style plot
ax.set(xlabel='PC1',
       ylabel='PC2')

#configure legend
plt.legend(title='Clusters',
           labels=['Cluster 0','Outliers']);

In [ ]:
# Determine Silhouette coefficients from DBSCAN 
arr_sil = silhouette_samples(
    X=df_features_selected, 
    labels=model.labels_, 
    metric="euclidean"
)

display("Silhouette coefficients for DBSCAN")

fig, ax = plt.subplots(figsize=[15, 15])
start = 0  
end = 0

for cluster in np.unique(model.labels_):
    mask = model_db.labels_ == cluster
    sv_len = len(arr_sil[mask])
    sv_sorted = np.sort(arr_sil[mask])
    end = end + sv_len
    ax.barh(range(start, end), width=sv_sorted, label=cluster)
    start = end
    
ax.legend()
plt.show()

### Insights:

DBSCAN only suggests one cluster and outliers. Although a few smaller clusters can be defined by reducing the distance to the nearest neighbors, the number of outliers is still very high.

## Alternative Calculation: Cosine-Similarity

### Function for Track Recommendation

In [ ]:
# Create Function to collect tracks identified as similar in a list
def track_recommendation(df_features_selected, row, sim_min):
    """
    Return labels of tracks in df_features_selected that are similar to the track with index `label`.

    Cosine similarity is used as the measure of likeness between two tracks.

    Args:
        df_features_selected (DataFrame): Numeric feature matrix (already scaled/engineered).
        label (hashable): Index label in df_features_selected of the reference track 
                          (e.g. 4000, 4001 oder eine Spotify-ID falls als Index gesetzt).
        sim_min (float): Minimum cosine similarity (between -1 and 1).

    Returns:
        recommendations (list[hashable]): Index labels of tracks with similarity >= sim_min.
    """
    columns = df_features_selected.columns
    recommendations = []

    v_ref = df_features_selected.loc[row, columns].to_numpy()

    for i in df_features_selected.index:
        if i == row:
            continue  # Skip self-match

        v_i = df_features_selected.loc[i, columns].to_numpy()
        dot_product  = np.dot(v_ref, v_i)
        norm_product = norm(v_ref) * norm(v_i)
        similarity   = dot_product / (norm_product + 1e-12)  # Protection against division by 0

        if similarity >= sim_min:
            recommendations.append(i)

    return recommendations

### Create Track Recommendations

In [ ]:
# Create recommendations based on a minimum similarity of 0.998
row = 73498
sim_min = 0.998

cs_selected_row = track_recommendation(df_features_selected, row, sim_min)

print(f"Number of similar Tracks:", len(cs_selected_row))

rows_with_ref = [row] + cs_selected_row 

display(df_filtered.loc[rows_with_ref])

### Insights:

Cosine similarity generates good suggestions and seems well suited to the task at hand.